# Direct Pipeline for Qwen 3.5, Gemma 4, and Aya Expanse

This notebook runs the same local Direct baseline for English-to-Chinese dialogue summarization across three Ollama models:

- `qwen3.5:9b`
- `gemma4:e4b`
- `aya-expanse:8b`

Each model uses the same prompt, the same fixed 100-sample test set, and the same output schema as the individual Direct baseline notebooks.


## 1. Model Setup

Make sure Ollama is installed and running before batch inference.

```bash
ollama serve
ollama pull qwen3.5:9b
ollama pull gemma4:e4b
ollama pull aya-expanse:8b
```

If your local model names are different, edit the `DIRECT_MODELS` list in the configuration cell.


In [ ]:
# Cell 1: Install required Python packages.
# Uncomment this cell if the packages are not installed in your notebook environment.

# %pip install requests pandas tqdm


In [ ]:
# Cell 2: Imports and global configuration

import json
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Set

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192
EXPECTED_SAMPLE_COUNT = 100


@dataclass(frozen=True)
class ModelConfig:
    label: str
    model_name: str
    output_subdir: str
    file_prefix: str

    @property
    def output_dir(self) -> Path:
        return PROJECT_ROOT / "outputs" / "direct" / self.output_subdir

    @property
    def checkpoint_path(self) -> Path:
        return self.output_dir / f"{self.file_prefix}_100samples_seed42_checkpoint.jsonl"

    @property
    def final_json_path(self) -> Path:
        return self.output_dir / f"{self.file_prefix}_100samples_seed42_results.json"

    @property
    def final_csv_path(self) -> Path:
        return self.output_dir / f"{self.file_prefix}_100samples_seed42_results.csv"

    @property
    def error_path(self) -> Path:
        return self.output_dir / f"{self.file_prefix}_100samples_seed42_errors.jsonl"


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root from a notebook or project working directory."""
    for path in [start, *start.parents]:
        if (path / "data" / "splits" / "test_100_seed42.json").exists():
            return path
    raise FileNotFoundError("Could not find data/splits/test_100_seed42.json")


PROJECT_ROOT = find_project_root()
DATASET_PATH = PROJECT_ROOT / "data" / "splits" / "test_100_seed42.json"

DIRECT_MODELS = [
    ModelConfig(
        label="qwen3.5",
        model_name="qwen3.5:9b",
        output_subdir="qwen3.5",
        file_prefix="direct_qwen9b",
    ),
    ModelConfig(
        label="gemma4",
        model_name="gemma4:e4b",
        output_subdir="gemma4",
        file_prefix="direct_gemma4_e4b",
    ),
    ModelConfig(
        label="aya_expanse",
        model_name="aya-expanse:8b",
        output_subdir="aya_expanse",
        file_prefix="direct_aya8b",
    ),
]

for config in DIRECT_MODELS:
    config.output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATASET_PATH)
for config in DIRECT_MODELS:
    print()
    print(f"[{config.label}] model:", config.model_name)
    print("Output directory:", config.output_dir)
    print("Checkpoint JSONL path:", config.checkpoint_path)
    print("Final JSON output path:", config.final_json_path)
    print("Final CSV output path:", config.final_csv_path)
    print("Error output path:", config.error_path)


In [ ]:
print(DATASET_PATH.exists())


In [ ]:
# Cell 3: Check whether Ollama is running and whether target models are available

def get_ollama_models() -> List[str]:
    response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
    response.raise_for_status()
    return [model.get("name", "") for model in response.json().get("models", [])]


def check_ollama_server(required_models: List[ModelConfig]) -> bool:
    try:
        downloaded_models = get_ollama_models()
        downloaded_set = set(downloaded_models)
        required_names = [config.model_name for config in required_models]
        missing = [name for name in required_names if name not in downloaded_set]

        print("Ollama server is running.")
        print("Downloaded models:", downloaded_models)

        if missing:
            print()
            print("Missing required models:", missing)
            print("Pull missing models before running batch inference:")
            for model_name in missing:
                print(f"  ollama pull {model_name}")
        else:
            print("All configured Direct models are available.")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server(DIRECT_MODELS)


In [ ]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()


## 2. Prompt Template


In [ ]:
# Cell 5: Direct prompt template
DIRECT_PROMPT = """Please summarize the following text in Chinese:

{dialogue}"""

print(DIRECT_PROMPT)


## 3. Shared Utilities


In [ ]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> Set[str]:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}


In [ ]:
# Cell 7: Load fixed 100-sample test set

def load_examples_from_sample_set(path: Path) -> List[Dict[str, Any]]:
    """Load the fixed 100-example sample shared across baseline runs."""
    if not path.exists():
        raise FileNotFoundError(f"Sample set not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    if not isinstance(raw_data, list):
        raise TypeError(f"Expected a top-level JSON array in {path}")

    examples = []

    for i, item in enumerate(raw_data):
        sample_index = item.get("sample_index", i)
        examples.append({
            "id": item.get("id", f"test100_seed42_{sample_index:05d}"),
            "sample_index": sample_index,
            "test_index": item.get("test_index", sample_index),
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_examples_from_sample_set(DATASET_PATH)

if len(test_data) != EXPECTED_SAMPLE_COUNT:
    raise ValueError(f"Expected {EXPECTED_SAMPLE_COUNT} examples, got {len(test_data)}")

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])


## 4. Direct Pipeline Functions


In [ ]:
# Cell 8: Direct agent and pipeline functions

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace named placeholders in the prompt."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def direct_agent(dialogue: str, config: ModelConfig) -> str:
    """Direct baseline: English dialogue -> Chinese summary."""
    prompt = fill_prompt(
        DIRECT_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=config.model_name,
        prompt=prompt,
        temperature=DEFAULT_TEMPERATURE,
    )

    return response.strip()


def run_direct_pipeline(example: Dict[str, Any], config: ModelConfig) -> Dict[str, Any]:
    """Run the Direct pipeline: English dialogue -> Chinese summary."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    final_chinese_summary = direct_agent(dialogue, config)

    return {
        "id": sample_id,
        "sample_index": example.get("sample_index", ""),
        "test_index": example.get("test_index", ""),
        "dialogue": dialogue,
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,
        "final_summary": final_chinese_summary,
        "pipeline": "direct",
        "model": config.model_name,
        "model_label": config.label,
        "num_model_calls": 1,
    }


## 5. Smoke Test


In [ ]:
# Cell 9: Run one example for each configured model

SMOKE_TEST_INDEX = 4
smoke_test_results = []

for config in DIRECT_MODELS:
    print(f"Running smoke test for {config.label} ({config.model_name})")
    result = run_direct_pipeline(test_data[SMOKE_TEST_INDEX], config)
    smoke_test_results.append(result)
    print("Generated summary:", result["final_summary"])
    print()


In [ ]:
# Cell 10: Inspect smoke-test outputs

def print_direct_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Direct Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Metadata ===")
    print("Pipeline:", result["pipeline"])
    print("Model label:", result["model_label"])
    print("Model:", result["model"])
    print("Model calls:", result["num_model_calls"])


for result in smoke_test_results:
    print_direct_result(result)
    print("=" * 80)


## 6. Optional Reset

Run this cell only when you want to delete previous checkpoint, final, and error outputs for all configured models.


In [ ]:
# Cell 11: Reset previous outputs for all configured models

RESET_OUTPUTS = False

if RESET_OUTPUTS:
    for config in DIRECT_MODELS:
        config.checkpoint_path.unlink(missing_ok=True)
        config.final_json_path.unlink(missing_ok=True)
        config.final_csv_path.unlink(missing_ok=True)
        config.error_path.unlink(missing_ok=True)
        print(f"[{config.label}] Previous output files reset.")
else:
    print("RESET_OUTPUTS is False. Existing checkpoints and outputs were preserved.")


## 7. Batch Inference with Checkpointing

This cell runs each model over the fixed 100-sample dataset. Each model has its own checkpoint file, so interrupted runs can resume independently.


In [ ]:
# Cell 12: Batch inference for all configured Direct models

MAX_EXAMPLES = len(test_data)
SLEEP_SECONDS = 0.2

subset = test_data[:MAX_EXAMPLES]

for config in DIRECT_MODELS:
    processed_ids = load_processed_ids(config.checkpoint_path)
    print()
    print(f"[{config.label}] Already processed: {len(processed_ids)} examples")

    for ex in tqdm(subset, desc=f"Running Direct pipeline: {config.label}"):
        sample_id = str(ex.get("id", "unknown"))

        if sample_id in processed_ids:
            continue

        try:
            record = run_direct_pipeline(ex, config)
            append_jsonl(record, config.checkpoint_path)
            processed_ids.add(sample_id)
            time.sleep(SLEEP_SECONDS)

        except Exception as e:
            error_record = {
                "id": sample_id,
                "sample_index": ex.get("sample_index", ""),
                "test_index": ex.get("test_index", ""),
                "model": config.model_name,
                "model_label": config.label,
                "error": repr(e),
                "dialogue": ex.get("dialogue", ""),
            }

            append_jsonl(error_record, config.error_path)
            print(f"[{config.label}] Error on {sample_id}: {repr(e)}")

    print(f"[{config.label}] Finished. Checkpoint outputs saved to: {config.checkpoint_path}")


## 8. Export Final Results to JSON and CSV

The per-model JSON and CSV outputs use the same schema as the mBART baseline: `sample_index`, `model_name`, `generated_summary_zh`, `reference_summary_zh`, `reference_summary_en`, and `dialogue`.

This cell also creates a combined CSV and JSON under `outputs/direct/all_models`.


In [ ]:
# Cell 13: Export per-model and combined Direct summaries

OUTPUT_FIELDNAMES = [
    "sample_index",
    "model_name",
    "generated_summary_zh",
    "reference_summary_zh",
    "reference_summary_en",
    "dialogue",
]


def sample_sort_key_value(value: Any) -> int:
    try:
        return int(value)
    except (TypeError, ValueError):
        return 10**9


def records_to_dataframe(records: List[Dict[str, Any]], config: ModelConfig) -> pd.DataFrame:
    rows = []

    for record in records:
        if "final_summary" not in record:
            continue

        rows.append({
            "sample_index": record.get("sample_index", ""),
            "model_name": record.get("model", config.model_name),
            "generated_summary_zh": record.get("final_summary", ""),
            "reference_summary_zh": record.get("reference_chinese_summary", ""),
            "reference_summary_en": record.get("reference_english_summary", ""),
            "dialogue": record.get("dialogue", ""),
        })

    df = pd.DataFrame(rows, columns=OUTPUT_FIELDNAMES)

    if not df.empty:
        df = df.drop_duplicates(subset=["sample_index"], keep="last")
        df = df.sort_values(by="sample_index", key=lambda series: series.map(sample_sort_key_value))

    return df


all_dfs = []

for config in DIRECT_MODELS:
    records = load_jsonl(config.checkpoint_path)
    df = records_to_dataframe(records, config)

    config.final_json_path.parent.mkdir(parents=True, exist_ok=True)
    json_records = df.to_dict(orient="records")

    with config.final_json_path.open("w", encoding="utf-8") as f:
        json.dump(json_records, f, ensure_ascii=False, indent=2)
        f.write("\n")

    df.to_csv(config.final_csv_path, index=False, encoding="utf-8")

    print(f"[{config.label}] Saved final JSON results to: {config.final_json_path}")
    print(f"[{config.label}] Saved final CSV results to: {config.final_csv_path}")

    if not df.empty:
        labeled_df = df.copy()
        labeled_df.insert(0, "model_label", config.label)
        all_dfs.append(labeled_df)

COMBINED_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "direct" / "all_models"
COMBINED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_JSON_PATH = COMBINED_OUTPUT_DIR / "direct_all_models_100samples_seed42_results.json"
COMBINED_CSV_PATH = COMBINED_OUTPUT_DIR / "direct_all_models_100samples_seed42_results.csv"

if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df = combined_df.sort_values(
        by=["model_label", "sample_index"],
        key=lambda series: series.map(sample_sort_key_value) if series.name == "sample_index" else series,
    )
else:
    combined_df = pd.DataFrame(columns=["model_label", *OUTPUT_FIELDNAMES])

with COMBINED_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(combined_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)
    f.write("\n")

combined_df.to_csv(COMBINED_CSV_PATH, index=False, encoding="utf-8")

print()
print("Saved combined JSON results to:", COMBINED_JSON_PATH)
print("Saved combined CSV results to:", COMBINED_CSV_PATH)
combined_df


## 9. Inspect Outputs


In [ ]:
# Cell 14: Compare generated Chinese summaries with the reference Chinese summaries

comparison_columns = [
    "model_label",
    "sample_index",
    "generated_summary_zh",
    "reference_summary_zh",
]

if not combined_df.empty:
    comparison_df = combined_df[comparison_columns].copy()
    display(comparison_df)
else:
    print("No combined results found.")


In [ ]:
# Cell 15: Inspect Direct outputs

inspection_columns = [
    "model_label",
    "sample_index",
    "model_name",
    "dialogue",
    "generated_summary_zh",
    "reference_summary_zh",
]

if not combined_df.empty:
    inspection_df = combined_df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")
